In [1]:
# import dependencies
import pandas as pd
import numpy as np
import geopandas as gpd
from functools import reduce

Note: Probably gonna rework how the National Risk Index is saved.

Technical debt for later.

In [2]:
# open csv file
df = pd.read_csv("../data/cleaned/town_level_merged_for_eda.csv")

# open geojson file
gdf = gpd.read_file("../data/cleaned/census.geojson")

In [3]:
# select GEOID and geometry columns, and rename NAMELSAD to town_name
gdf = gdf[["GEOID", "geometry", "NAMELSAD"]]
gdf = gdf.rename(columns={"NAMELSAD": "town_name"})

# export to geojson
gdf.to_file("../docs/static/resources/town_boundaries.geojson", driver="GeoJSON")

In [4]:
# data cleaning and preprocessing

# convert town_area_sqm to area_sq_km
df["area_sq_km"] = df["town_area_sqm"] / 1e6

# cap extreme outliers in poverty variable (winsorize)
df["pct_below_poverty"] = df["pct_below_poverty"].clip(
    upper=df["pct_below_poverty"].quantile(0.99)
)

# fill funding and claims NaNs with 0
cols_to_fill = [
    "federalShareObligated_adj",
    "funding_per_capita",
    "log_funding_per_capita",
    "funding_per_occupied_unit",
    "log_funding_per_occupied_unit",
    "claims_paid_per_capita",
    "current_insurance_penetration",
]
df[cols_to_fill] = df[cols_to_fill].fillna(0)

# mark towns with no population
df["valid_population"] = df["total_population"] > 0
df_zero_pop = df[~df["valid_population"]].copy()
df_valid = df[df["valid_population"]].copy()

In [5]:
# functions for building indices


def rank_normalize(df, cols):
    """
    Apply rank-based normalization (percentile ranks) to specified columns in a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        cols (list): List of column names to normalize.

    Returns:
        pd.DataFrame: DataFrame with specified columns rank-normalized.
    """
    df = df.copy()
    for col in cols:
        df[col] = df[col].rank(pct=True)
    return df


def build_index(df, risk_vars, vuln_vars, funding_var):
    """
    Build risk, vulnerability, and need indices, and compute funding gap.

    Args:
        df (pd.DataFrame): Input DataFrame.
        risk_vars (list): List of risk variable column names.
        vuln_vars (list): List of vulnerability variable column names.
        funding_var (str): Column name for funding variable.

    Returns:
        pd.DataFrame: DataFrame with new index and gap columns.
    """
    df = df.copy()

    all_vars = risk_vars + vuln_vars

    # normalize risk and vulnerability variables
    if all_vars:
        df_norm = rank_normalize(df, all_vars)
    else:
        df_norm = df.copy()

    # calculate risk and vulnerability indices as the mean of their respective (z-scored) variables
    df["risk_index"] = df_norm[risk_vars].mean(axis=1) if risk_vars else 0
    df["vuln_index"] = df_norm[vuln_vars].mean(axis=1) if vuln_vars else 0

    # combined need index (normalized 0–1-ish if rank, centered if z)
    df["need_index"] = (df["risk_index"] + df["vuln_index"]) / 2

    # # scale funding variable to 0–1 range using rank normalization, so it's on the same scale as the need index
    # # safeguard - I've already log-transformed funding and filled NaNs in preprocessing
    funding = df[funding_var].fillna(0).copy()
    df["funding_scaled"] = funding.rank(pct=True)

    # gap: positive => overfunded relative to need; negative => underfunded
    df["gap_index"] = df["funding_scaled"] - df["need_index"]

    return df

In [6]:
# function to assign quadrants based on need and funding indices


def add_quadrants(df):
    """
    Add quadrant labels based on need and funding indices.
        Quadrants:
        - zero_funding: no funding regardless of need
        - underfunded: high need, low funding
        - aligned: high need, high funding
        - overfunded: low need, high funding
        - low_priority: low need, low funding

    Args:
        df (pd.DataFrame): Input DataFrame with 'need_index' and 'funding_scaled' columns.

    Returns:
        pd.DataFrame: DataFrame with an additional 'quadrant' column.
    """
    df = df.copy()

    # calculate median for need to define quadrants
    need_med = df["need_index"].median()

    df["quadrant"] = "unassigned"

    # zero funding = separate category regardless of need
    df.loc[df["funding_scaled"] == df["funding_scaled"].min(), "quadrant"] = (
        "zero_funding"
    )

    # get mask for non-zero funding to apply quadrant logic only to those rows
    mask = df["funding_scaled"] > df["funding_scaled"].min()

    # calculate median of non-zero funding for quadrant split
    fund_med_nonzero = df.loc[mask, "funding_scaled"].median()

    # assign quadrants based on need and funding relative to their medians
    df.loc[
        mask
        & (df["need_index"] >= need_med)
        & (df["funding_scaled"] < fund_med_nonzero),
        "quadrant",
    ] = "underfunded"
    df.loc[
        mask
        & (df["need_index"] >= need_med)
        & (df["funding_scaled"] >= fund_med_nonzero),
        "quadrant",
    ] = "aligned"
    df.loc[
        mask
        & (df["need_index"] < need_med)
        & (df["funding_scaled"] >= fund_med_nonzero),
        "quadrant",
    ] = "overfunded"
    df.loc[
        mask
        & (df["need_index"] < need_med)
        & (df["funding_scaled"] < fund_med_nonzero),
        "quadrant",
    ] = "low_priority"

    return df

In [7]:
def analyze_models(df, model_specs):
    """
    Run the full analysis for each model specification, including index building, quadrant assignment, and result summarization.

    Args:
        df (pd.DataFrame): Input DataFrame with town-level data.
        model_specs (dict): Dictionary of model specifications with risk and vulnerability variable lists.

    Returns:
        dict: Dictionary containing results for each model specification.
    """
    # initialize results dictionary
    results = {}

    # loop through model specifications and run analysis
    for name, spec in model_specs.items():
        # extract variable lists from spec
        risk_vars = spec.get("risk", [])
        vuln_vars = spec.get("vuln", [])
        funding_var = "log_funding_per_capita"

        # drop any town with no population to avoid skewing indices
        df_valid = df[df["valid_population"]].copy()

        # build indices and assign quadrants
        df_results = build_index(df_valid, risk_vars, vuln_vars, funding_var)
        df_quadrants = add_quadrants(df_results)

        # store results in dictionary
        results[name] = {
            "df_results": df_results,
            "quadrants": df_quadrants,
        }

    # return the results dictionary
    return results

In [8]:
# select model specifications of interest
model_specs = {
    # single risk variable with core vulnerability variables
    "core_EAL_model": {
        "risk": ["IFLD_EALT_weighted"],
        "vuln": ["pct_below_poverty", "percent_elderly", "pct_no_vehicle"],
    },
    # EAL per capita
    "eal_per_capita_model": {
        "risk": ["EAL_per_capita"],
        "vuln": ["pct_below_poverty", "percent_elderly", "pct_no_vehicle"],
    },
    # FEMA's own risk score, for benchmarking
    "fema_national_risk_index": {
        "risk": ["RISK_SCORE_avg"],
    },
}

In [9]:
# run models
results = analyze_models(df_valid, model_specs)

In [10]:
# collect base columns for final DataFrame and rename for clarity
base_cols = [
    "GEOID",
    "town_name",
    "total_population",
    "area_sq_km",
    "pct_river_corridor",
    "valid_population",
    "has_funding",
    "federalShareObligated_adj",
    "funding_per_capita",
    "pct_below_poverty",
    "percent_elderly",
    "pct_no_vehicle",
]
base_df = df_valid[base_cols].copy()
base_df = base_df.rename(
    columns={
        "total_population": "population",
        "federalShareObligated_adj": "funding_total",
    }
)

# rename model-specific columns from results
model_map = {
    "core_EAL_model": "eal",
    "eal_per_capita_model": "eal_per_capita",
    "fema_national_risk_index": "nri",
}

# iterate through models, rename columns, compute ranks, and append to list for merging
model_dfs = []
for model_key, suffix in model_map.items():
    res = results[model_key]["df_results"].copy()
    quad = results[model_key]["quadrants"][["GEOID", "quadrant"]].copy()
    res = res.merge(quad, on="GEOID", how="left").rename(
        columns={
            "risk_index": f"risk_{suffix}",
            "need_index": f"need_{suffix}",
            "gap_index": f"gap_{suffix}",
            "funding_scaled": f"funding_rank_{suffix}",
            "quadrant": f"quadrant_{suffix}",
        }
    )
    # compute ranks
    res[f"risk_rank_{suffix}"] = res[f"risk_{suffix}"].rank(ascending=False, pct=True)
    res[f"need_rank_{suffix}"] = res[f"need_{suffix}"].rank(ascending=False, pct=True)
    res[f"gap_rank_{suffix}"] = res[f"gap_{suffix}"].rank(ascending=True, pct=True)
    # select relevant columns
    model_dfs.append(
        res[
            [
                "GEOID",
                f"risk_{suffix}",
                f"need_{suffix}",
                f"gap_{suffix}",
                f"risk_rank_{suffix}",
                f"need_rank_{suffix}",
                f"gap_rank_{suffix}",
                f"quadrant_{suffix}",
            ]
        ]
    )

# merge all model DataFrames on GEOID
# reduce applies the merge function cumulatively to the list of DataFrames, merging them one by one on GEOID
df_merged = reduce(lambda left, right: pd.merge(left, right, on="GEOID"), model_dfs)
df_merged = pd.merge(base_df, df_merged, on="GEOID", how="left")

# compute composite vulnerability index (always the same across all models)
df_merged["vulnerability_index"] = (
    df_merged[["pct_below_poverty", "percent_elderly", "pct_no_vehicle"]]
    .rank(pct=True)
    .mean(axis=1)
)
df_merged["vulnerability_rank"] = df_merged["vulnerability_index"].rank(
    ascending=False, pct=True
)

# compute funding ranks for the final DataFrame (also not model-specific)
df_merged["funding_rank"] = (
    df_merged["funding_total"]
    .where(df_merged["funding_total"] > 0)
    .rank(pct=True)
    .fillna(0)
)


# relative scaling function
def to_relative_centered(series):
    mean = series.mean()
    return (series / mean) - 1  # centered at 0


# compute relative to state average columns
rel_cols = [
    "risk_eal",
    "risk_eal_per_capita",
    "risk_nri",
    "need_eal",
    "need_eal_per_capita",
    "need_nri",
    "vulnerability_index",
]

for col in rel_cols:
    df_merged[f"{col}_rel"] = to_relative_centered(df_merged[col])

# compute funding (log before computing relative to reduce skew)
df_merged["funding_log"] = np.log1p(df_merged["funding_total"])
df_merged["funding_rel"] = to_relative_centered(df_merged["funding_log"])

# compute standardized gap scores for better comparability across models (since the gap can be positive or negative)
for col in ["gap_eal", "gap_eal_per_capita", "gap_nri"]:
    # df_merged[f"{col}_rel"] = df_merged[col]/ abs(df_merged[col]).mean()
    df_merged[f"{col}_rel"] = df_merged[col] / df_merged[col].abs().max()

# format funding_per_capita and population for better readability in the web app
df_merged["funding_per_capita_fmt"] = df_merged["funding_per_capita"].round(0)
df_merged["population"] = df_merged["population"].astype(int)

# reorder columns
final_cols = [
    "GEOID",
    "town_name",
    "population",
    "valid_population",
    "area_sq_km",
    "pct_river_corridor",
    # raw
    "risk_eal",
    "risk_eal_per_capita",
    "risk_nri",
    "need_eal",
    "need_eal_per_capita",
    "need_nri",
    "gap_eal",
    "gap_eal_per_capita",
    "gap_nri",
    "funding_total",
    "funding_per_capita",
    # relative
    "risk_eal_rel",
    "risk_eal_per_capita_rel",
    "risk_nri_rel",
    "need_eal_rel",
    "need_eal_per_capita_rel",
    "need_nri_rel",
    "gap_eal_rel",
    "gap_eal_per_capita_rel",
    "gap_nri_rel",
    "funding_rel",
    "vulnerability_index_rel",
    # ranks
    "funding_rank",
    "vulnerability_rank",
    "risk_rank_eal",
    "risk_rank_eal_per_capita",
    "risk_rank_nri",
    "need_rank_eal",
    "need_rank_eal_per_capita",
    "need_rank_nri",
    "gap_rank_eal",
    "gap_rank_eal_per_capita",
    "gap_rank_nri",
    # categorical
    "quadrant_eal",
    "quadrant_eal_per_capita",
    "quadrant_nri",
]

# ensure all final columns are present in the merged DataFrame before selecting
final_cols = [col for col in final_cols if col in df_merged.columns]
df_final = df_merged[final_cols]


# append zero population towns back to the final DataFrame with nulls for indices and gaps
df_zero_pop = df_zero_pop[base_cols].copy()
derived_cols = [col for col in df_final.columns if col not in base_cols]
df_zero_pop[derived_cols] = None
df_zero_pop["population"] = df_zero_pop["population"].fillna(0).astype(int)
df_final = pd.concat([df_final, df_zero_pop[final_cols]], ignore_index=True)

C:\Users\johbr\AppData\Local\Temp\ipykernel_16916\4199562820.py:181: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_zero_pop["population"] = df_zero_pop["population"].fillna(0).astype(int)
C:\Users\johbr\AppData\Local\Temp\ipykernel_16916\4199562820.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final, df_zero_pop[final_cols]], ignore_index=True)


In [11]:
null_counts = df_final.isnull().sum()
print("Null counts in final DataFrame:")
print(null_counts)

Null counts in final DataFrame:
GEOID                       0
town_name                   0
population                  0
valid_population            0
area_sq_km                  0
pct_river_corridor          0
risk_eal                    6
risk_eal_per_capita         6
risk_nri                    6
need_eal                    6
need_eal_per_capita         6
need_nri                    6
gap_eal                     6
gap_eal_per_capita          6
gap_nri                     6
funding_total               6
funding_per_capita          0
risk_eal_rel                6
risk_eal_per_capita_rel     6
risk_nri_rel                6
need_eal_rel                6
need_eal_per_capita_rel     6
need_nri_rel                6
gap_eal_rel                 6
gap_eal_per_capita_rel      6
gap_nri_rel                 6
funding_rel                 6
vulnerability_index_rel     6
funding_rank                6
vulnerability_rank          6
risk_rank_eal               6
risk_rank_eal_per_capita    6
risk_ran

In [12]:
# check final DataFrame
pd.set_option("display.max_columns", None)
display(df_final.describe(include="all"))
pd.reset_option("display.max_columns")

,GEOID,town_name,population,valid_population,area_sq_km,pct_river_corridor,risk_eal,risk_eal_per_capita,risk_nri,need_eal,need_eal_per_capita,need_nri,gap_eal,gap_eal_per_capita,gap_nri,funding_total,funding_per_capita,risk_eal_rel,risk_eal_per_capita_rel,risk_nri_rel,need_eal_rel,need_eal_per_capita_rel,need_nri_rel,gap_eal_rel,gap_eal_per_capita_rel,gap_nri_rel,funding_rel,vulnerability_index_rel,funding_rank,vulnerability_rank,risk_rank_eal,risk_rank_eal_per_capita,risk_rank_nri,need_rank_eal,need_rank_eal_per_capita,need_rank_nri,gap_rank_eal,gap_rank_eal_per_capita,gap_rank_nri,quadrant_eal,quadrant_eal_per_capita,quadrant_nri
count,2.560000e+02,256,256.000000,256,256.000000,256.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,2.500000e+02,2.500000e+02,250.000000,2.500000e+02,256.000000,2.500000e+02,250.000000,2.500000e+02,2.500000e+02,2.500000e+02,2.500000e+02,250.000000,2.500000e+02,250.000000,2.500000e+02,2.500000e+02,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250,250,250
unique,NaN,256,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,5,5
top,NaN,Addison town,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,zero_funding,zero_funding,zero_funding
freq,NaN,1,NaN,250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,130,130,130
mean,5.001514e+09,NaN,2527.757812,NaN,97.278278,3.512886,0.502000,0.502000,0.502000,0.502000,0.502000,0.251000,4.884981e-18,1.221245e-17,0.251000,3.099348e+05,155.932123,-5.329071e-18,0.000000,-1.065814e-17,3.552714e-18,8.437695e-18,-1.065814e-17,0.000000,7.105427e-18,0.262552,2.842171e-17,2.184919e-16,0.242000,0.502000,0.502000,0.502000,0.502000,0.502000,0.502000,0.502000,0.502000,0.502000,0.502000,NaN,NaN,NaN
std,8.685967e+05,NaN,3973.365068,NaN,34.758921,2.510260,0.289252,0.289242,0.289235,0.186213,0.177312,0.144618,2.723067e-01,2.922798e-01,0.291383,6.725437e+05,377.820867,5.761990e-01,0.576179,5.761655e-01,3.709426e-01,3.532106e-01,5.761655e-01,0.396948,4.242088e-01,0.304794,1.059123e+00,3.710306e-01,0.322268,0.289249,0.289252,0.289242,0.289235,0.289251,0.289251,0.289235,0.289251,0.289252,0.289242,NaN,NaN,NaN
min,5.000100e+09,NaN,0.000000,NaN,3.813185,0.000000,0.004000,0.004000,0.004000,0.034333,0.132000,0.002000,-6.373333e-01,-6.066667e-01,-0.230000,0.000000e+00,0.000000,-9.920319e-01,-0.992032,-9.920319e-01,-9.316069e-01,-7.370518e-01,-9.920319e-01,-0.929057,-8.805031e-01,-0.240586,-1.000000e+00,-9.508632e-01,0.000000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,0.004000,NaN,NaN,NaN
25%,5.000731e+09,NaN,784.750000,NaN,82.562679,1.921324,0.253000,0.251500,0.253000,0.358500,0.356833,0.126500,-1.943333e-01,-2.097500e-01,0.039000,0.000000e+00,0.000000,-4.960159e-01,-0.499004,-4.960159e-01,-2.858566e-01,-2.891766e-01,-4.960159e-01,-0.283285,-3.044267e-01,0.040795,-1.000000e+00,-2.629482e-01,0.000000,0.254000,0.253000,0.258000,0.256000,0.253500,0.253000,0.256000,0.253000,0.253000,0.258000,NaN,NaN,NaN
50%,5.001711e+09,NaN,1330.000000,NaN,102.586830,2.985074,0.502000,0.503000,0.502000,0.489667,0.498333,0.251000,-4.400000e-02,-1.650000e-02,0.205500,0.000000e+00,0.000000,0.000000e+00,0.001992,0.000000e+00,-2.456839e-02,-7.304117e-03,0.000000e+00,-0.064140,-2.394775e-02,0.214958,-1.000000e+00,3.054449e-02,0.000000,0.502000,0.502000,0.501000,0.502000,0.502000,0.502000,0.502000,0.502000,0.502000,0.502000,NaN,NaN,NaN
75%,5.002307e+09,NaN,2806.250000,NaN,116.306491,4.311916,0.751000,0.746000,0.748000,0.636583,0.638333,0.374000,1.938333e-01,2.301667e-01,0.475000,3.289318e+05,135.828202,4.960159e-01,0.486056,4.900398e-01,2.680943e-01,2.715803e-01,4.900398e-01,0.282556,3.340590e-01,0.496862

In [13]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256 entries, 0 to 255
Data columns (total 42 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   GEOID                     256 non-null    int64  
 1   town_name                 256 non-null    object 
 2   population                256 non-null    int32  
 3   valid_population          256 non-null    bool   
 4   area_sq_km                256 non-null    float64
 5   pct_river_corridor        256 non-null    float64
 6   risk_eal                  250 non-null    float64
 7   risk_eal_per_capita       250 non-null    float64
 8   risk_nri                  250 non-null    float64
 9   need_eal                  250 non-null    float64
 10  need_eal_per_capita       250 non-null    float64
 11  need_nri                  250 non-null    float64
 12  gap_eal                   250 non-null    float64
 13  gap_eal_per_capita        250 non-null    float64
 14  gap_nri   

In [14]:
df_final.columns

Index(['GEOID', 'town_name', 'population', 'valid_population', 'area_sq_km',
       'pct_river_corridor', 'risk_eal', 'risk_eal_per_capita', 'risk_nri',
       'need_eal', 'need_eal_per_capita', 'need_nri', 'gap_eal',
       'gap_eal_per_capita', 'gap_nri', 'funding_total', 'funding_per_capita',
       'risk_eal_rel', 'risk_eal_per_capita_rel', 'risk_nri_rel',
       'need_eal_rel', 'need_eal_per_capita_rel', 'need_nri_rel',
       'gap_eal_rel', 'gap_eal_per_capita_rel', 'gap_nri_rel', 'funding_rel',
       'vulnerability_index_rel', 'funding_rank', 'vulnerability_rank',
       'risk_rank_eal', 'risk_rank_eal_per_capita', 'risk_rank_nri',
       'need_rank_eal', 'need_rank_eal_per_capita', 'need_rank_nri',
       'gap_rank_eal', 'gap_rank_eal_per_capita', 'gap_rank_nri',
       'quadrant_eal', 'quadrant_eal_per_capita', 'quadrant_nri'],
      dtype='object')

In [15]:
# export final DataFrame to CSV for web app
df_final.to_csv("../docs/static/resources/town_stats.csv", index=False)